In [62]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage,HumanMessage,ToolMessage
from langchain_core.prompts import SystemMessagePromptTemplate,HumanMessagePromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda,RunnablePassthrough,RunnableSequence,RunnableParallel,RouterRunnable

In [63]:
load_dotenv(override=True)

True

In [64]:
llm = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

In [65]:
result = llm.invoke("write very very small poem in english")
result.content

'Moonlight hums.  \nNight breathes soft.'

In [66]:
system_message = SystemMessage("You are a helpful chef.The user is {diet}.")
user_message = HumanMessage("Give me a recipe for {dish}")

system_prompt = SystemMessagePromptTemplate.from_template(system_message.content)
user_prompt = HumanMessagePromptTemplate.from_template(user_message.content)

# dynamic_system = ChatPromptTemplate.from_messages([system_prompt])
# dynamic_user = ChatPromptTemplate.from_messages([user_prompt])
# print(dynamic_system.invoke({"diet": "vegan"}))
# print(dynamic_user.invoke({"dish": "Banoffee Pie"}))

dynamic_template = ChatPromptTemplate([system_prompt,user_prompt])
quection = dynamic_template.invoke({"diet": "vegan","dish": "Banoffee Pie"})
# messages = [system_message,user_message]
result = llm.invoke(quection)
print(result.content)

Absolutely — here’s a **vegan banoffee pie** recipe.

## Vegan Banoffee Pie

### Serves
8

### Time
- Prep: 25 minutes
- Chill: 4+ hours

### Ingredients

#### Crust
- 300 g digestive biscuits or graham crackers
- 120 g vegan butter, melted

#### Toffee layer
- 1 can coconut condensed milk or vegan condensed milk
- 2 tbsp brown sugar
- 2 tbsp vegan butter
- Pinch of salt

#### Filling
- 3–4 ripe bananas, sliced
- 250 ml vegan whipping cream, chilled
- 1–2 tbsp icing sugar, optional
- 1 tsp vanilla extract, optional

#### Topping
- Cocoa powder or grated dark vegan chocolate

### Instructions

1. **Make the crust**
   - Crush the biscuits into fine crumbs.
   - Mix with the melted vegan butter until evenly combined.
   - Press firmly into a pie dish, covering the base and sides.
   - Chill in the fridge while you make the filling.

2. **Make the toffee**
   - In a saucepan over medium heat, combine the coconut condensed milk, brown sugar, vegan butter, and salt.
   - Stir constantly for

In [67]:
system_message = SystemMessage("You are a helpful chef.The user is {diet}.")
user_message = HumanMessage("Give me a recipe for {dish}")

system_prompt = SystemMessagePromptTemplate.from_template(system_message.content)
user_prompt = HumanMessagePromptTemplate.from_template(user_message.content)

# dynamic_system = ChatPromptTemplate.from_messages([system_prompt])
# dynamic_user = ChatPromptTemplate.from_messages([user_prompt])
# print(dynamic_system.invoke({"diet": "vegan"}))
# print(dynamic_user.invoke({"dish": "Banoffee Pie"}))


dynamic_template = ChatPromptTemplate([system_prompt,user_prompt])

recipe_chain  = dynamic_template | llm | parser 

# result = recipe_chain.invoke({"diet": "vegan","dish": "Banoffee Pie"})

print(result)

content='Absolutely — here’s a **vegan banoffee pie** recipe.\n\n## Vegan Banoffee Pie\n\n### Serves\n8\n\n### Time\n- Prep: 25 minutes\n- Chill: 4+ hours\n\n### Ingredients\n\n#### Crust\n- 300 g digestive biscuits or graham crackers\n- 120 g vegan butter, melted\n\n#### Toffee layer\n- 1 can coconut condensed milk or vegan condensed milk\n- 2 tbsp brown sugar\n- 2 tbsp vegan butter\n- Pinch of salt\n\n#### Filling\n- 3–4 ripe bananas, sliced\n- 250 ml vegan whipping cream, chilled\n- 1–2 tbsp icing sugar, optional\n- 1 tsp vanilla extract, optional\n\n#### Topping\n- Cocoa powder or grated dark vegan chocolate\n\n### Instructions\n\n1. **Make the crust**\n   - Crush the biscuits into fine crumbs.\n   - Mix with the melted vegan butter until evenly combined.\n   - Press firmly into a pie dish, covering the base and sides.\n   - Chill in the fridge while you make the filling.\n\n2. **Make the toffee**\n   - In a saucepan over medium heat, combine the coconut condensed milk, brown sugar

In [68]:
# shoping_list_message = HumanMessage("Make a shopping list from this recipe:\n{recipe}")
# shoping_list_prompt = HumanMessagePromptTemplate.from_template(shoping_list_message.content)
# shoping_list_template = ChatPromptTemplate.from_messages([shoping_list_prompt])

# shoping_list_chain = shoping_list_template | llm | parser
# output = shoping_list_chain.invoke( {"recipe":result} )
# print()


In [74]:
from operator import itemgetter

In [75]:

shoping_list_message = HumanMessage("Make a shopping list from this recipe:\n{recipe} only list {count} short")
shoping_list_prompt = HumanMessagePromptTemplate.from_template(shoping_list_message.content)
shoping_list_template = ChatPromptTemplate.from_messages([shoping_list_prompt])
# shoping_list_prompt = ChatPromptTemplate.from_template('''Make a shopping list from this recipe:\n{recipe}''')

composed_chain = {"recipe":recipe_chain,"count":itemgetter("count")} | shoping_list_template | llm | parser
output = composed_chain.invoke( {"diet": "vegan","dish": "Banoffee Pie","count":5} )
print(output)

- Digestive biscuits or graham crackers
- Vegan butter
- Full-fat coconut milk
- Brown sugar
- Vanilla extract
- Salt
- Bananas
- Vegan whipping cream
- Icing sugar
- Cocoa powder or dark chocolate


In [92]:
from datetime import date, timedelta

In [ ]:
system_movies_prompt = SystemMessagePromptTemplate.from_template("""You are a movie recommender.
Suggest exactly 1 film that matches the user's mood, in {language}.
Give: title (year) and one line on why it fits the mood.
If you are not sure a film was released in that period, say so instead of guessing.
Output a numbered list. No introduction.""")

system_snacks_prompt = SystemMessagePromptTemplate.from_template("""You are a snack planner for movie nights.
Suggest exactly 3 snacks that go well with the user's mood.
Prefer snacks that are easy to make or buy in India.
For each give the snack and one short reason.
Output a numbered list. No introduction.""")

system_playlist_prompt = SystemMessagePromptTemplate.from_template("""You are a music curator.
The user chose this film:
{movies}

List exactly 3 songs from this film's own soundtrack.
Rules:
- Only songs that are actually in this film
- All songs must be in {language}
- For each give: song title, singer or music director, and a few words on the feel of the song
- If you are not sure of the film's songs, reply exactly: "I am not sure about this film's songs."
- Never use songs from other films
Output a numbered list. No introduction.""")

user_prompt = HumanMessagePromptTemplate.from_template("""
Mood: {mood}
Language: {language}
""")

movies_template = ChatPromptTemplate.from_messages([system_movies_prompt,user_prompt])
snacks_template = ChatPromptTemplate.from_messages([system_snacks_prompt,user_prompt])
playlist_template = ChatPromptTemplate.from_messages([system_playlist_prompt,user_prompt])

# movies_template = movies_template.partial(
#     today=lambda: date.today().isoformat(),
#     start_date=lambda: (date.today() - timedelta(days=765)).isoformat(),
# )

movies_chain = movies_template | llm | parser
snacks_chain = snacks_template | llm | parser
playlist_chain = playlist_template | llm | parser

parallel_chain = RunnableParallel(
   movies= movies_chain ,
    snacks=snacks_chain 
)

pipeline = (
    RunnablePassthrough.assign(
        movies= movies_chain ,
        snacks=snacks_chain ,
    )  | RunnablePassthrough.assign(songs=playlist_chain)
)
output = pipeline.invoke({"mood":"cozy and relaxed","language":"Telugu"})
for name, text in output.items():
    print(f"=== {name.upper()} ===")
    print(text, "\n")


=== MOOD ===
cozy and relaxed 

=== LANGUAGE ===
Telugu 

=== MOVIES ===
1. **పెళ్లిచూపులు (2016)** — తేలికైన హాస్యం, సున్నితమైన భావోద్వేగాలు, ఆహ్లాదకరమైన వాతావరణంతో సేదతీరే moodకి బాగా సరిపోతుంది. 

=== SNACKS ===
1. **వెన్నతో పాప్‌కార్న్** — తేలికగా తయారవుతుంది, సినిమా చూస్తూ సాఫీగా తినడానికి బాగా సరిపోతుంది.  
2. **మసాలా చనా** — గట్టిగా తినగలిగే, కాస్త క్రంచీగా ఉండే స్నాక్‌గా cozy mood కి చక్కగా ఉంటుంది.  
3. **బిస్కెట్లు + వేడి టీ** — రిలాక్స్‌గా కూర్చొని సినిమా ఆస్వాదించడానికి చాలా సింపుల్‌గా, comforting‌గా ఉంటుంది. 

=== SONGS ===
1. **సునోనా సునైనా** — **సింగర్: రాహుల్ రవీంద్రన్, సునీత** — మృదువైన, ప్రేమభరితమైన, చాలా సాఫ్ట్‌గా సాగే పాట.  
2. **పెళ్లిచూపులు** — **సింగర్: విజయ్ ప్రకాష్** — ఆహ్లాదకరమైన, లైట్‌గా ఉండే, రిలాక్స్‌డ్ ఫీల్ ఇచ్చే ట్రాక్.  
3. **మీసాల పిల్ల** — **సింగర్: డిలీప్** — సరదాగా, చురుకుగా, మంచి పాజిటివ్ ఎనర్జీ ఉన్న పాట. 

